# Prerequisites

In [1]:
# Limit number of cores
import os
os.nice(19)
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

# Create logger
import logging
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

if not logger.hasHandlers():
    ch = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    ch.setFormatter(formatter)
    logger.addHandler(ch)

In [2]:
import pyccl as ccl
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import h5py
from getdist import loadMCSamples, plots

rootpath = '/home/dneup16/leiden_phd/scripts/GNAT'
if rootpath not in os.sys.path:
    os.sys.path.append(rootpath)

import src.galaxy_alignment_prediction_tool.multipoles
import src.galaxy_alignment_prediction_tool.powerspectrum
import src.galaxy_alignment_prediction_tool.projections
import src.galaxy_alignment_prediction_tool.fitter
import src.galaxy_alignment_prediction_tool.utils

# Reload to update changes
import importlib
importlib.reload(src.galaxy_alignment_prediction_tool.multipoles)
importlib.reload(src.galaxy_alignment_prediction_tool.powerspectrum)
importlib.reload(src.galaxy_alignment_prediction_tool.projections)
importlib.reload(src.galaxy_alignment_prediction_tool.fitter)
importlib.reload(src.galaxy_alignment_prediction_tool.utils)

from src.galaxy_alignment_prediction_tool.multipoles import multipoles
from src.galaxy_alignment_prediction_tool.powerspectrum import powerSpectrum
from src.galaxy_alignment_prediction_tool.projections import projections
from src.galaxy_alignment_prediction_tool.fitter import fitter
from src.galaxy_alignment_prediction_tool.utils import ioUtils

# Initialise classes
fitterHandler = fitter(logger=logger)
ioUtilsHandler = ioUtils(logger=logger)
powerSpectrumHandler = powerSpectrum()
multipolesHandler = multipoles()
projectionsHandler = projections()

# Helper functions

In [3]:
def create_contour_comparison_plot(
    snapshot, 
    sim, 
    selection, 
    estimator,
    figpath,
    respath,
    redshift_dict,
    param_names=None,
) -> None:
    filepath_vanilla = f'{respath}run_20260311_bugfix/{selection}/fit_results_{estimator}_{sim}/fit_results_Snapshot_{snapshot}'
    filepath_NL_scaled = f'{respath}run_20260311_NL_scaling/{selection}/fit_results_{estimator}_{sim}/fit_results_Snapshot_{snapshot}'

    samples_vanilla = loadMCSamples(f'{filepath_vanilla}/mcmc')
    samples_nl = loadMCSamples(f'{filepath_NL_scaled}/mcmc')

    if param_names is None:
        param_names = [param.name for param in samples_nl.getParamNames().names]

    g = plots.get_subplot_plotter(
        subplot_size=2.5,
    )
    g.triangle_plot([samples_vanilla, samples_nl], legend_labels=['Vanilla', 'NL Scaled'], params=param_names)
    plt.suptitle(f'Snapshot {snapshot} (z={redshift_dict.get(str(snapshot), "Unknown")}) - {estimator} - {sim} - {selection}', y=1.03)
    plt.savefig(f'{figpath}comparison_mcmc_Snapshot_{selection}_{estimator}_{sim}_{snapshot}.png', bbox_inches='tight')
    plt.close()

def create_redshift_evolution_comparison_plot(
    sim, 
    selection, 
    estimator,
    figpath,
    respath,
):

    catpath_vanilla = f'{respath}run_20260311_bugfix/IA_fitting_results_summary_{selection}.csv'
    catpath_nl_scaled = f'{respath}run_20260311_NL_scaling/IA_fitting_results_summary_{selection}.csv'

    df_vanilla = pd.read_csv(catpath_vanilla, sep='\t')
    df_nl_scaled = pd.read_csv(catpath_nl_scaled, sep='\t')

    df_vanilla_cut = df_vanilla[(df_vanilla['simulation'] == sim) & (df_vanilla['estimator'] == estimator)]
    df_nl_scaled_cut = df_nl_scaled[(df_nl_scaled['simulation'] == sim) & (df_nl_scaled['estimator'] == estimator)]

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].errorbar(df_vanilla_cut['redshift'], df_vanilla_cut['A_IA'], yerr=df_vanilla_cut['A_IA_err'], fmt='o', label='Vanilla')
    axes[0].errorbar(df_nl_scaled_cut['redshift'], df_nl_scaled_cut['A_IA'], yerr=df_nl_scaled_cut['A_IA_err'], fmt='s', label='NL Scaled')
    axes[0].set_xlabel('Redshift')
    axes[0].set_ylabel(r'$A_{\rm IA}$')
    axes[0].legend(loc='upper left')

    axes[1].errorbar(df_vanilla_cut['redshift'], df_vanilla_cut['b_g'], yerr=df_vanilla_cut['b_g_err'], fmt='o', label='Vanilla')
    axes[1].errorbar(df_nl_scaled_cut['redshift'], df_nl_scaled_cut['b_g'], yerr=df_nl_scaled_cut['b_g_err'], fmt='s', label='NL Scaled')
    axes[1].set_xlabel('Redshift')
    axes[1].set_ylabel(r'$b_{\rm g}$')
    axes[1].legend(loc='upper left')

    plt.suptitle(f'Comparison of IA fitting results - {estimator} - {sim} - {selection}')
    outpath = f'{figpath}/comparison_redshift_evolution_{selection}_{estimator}_{sim}.png'
    plt.savefig(outpath, bbox_inches='tight')
    plt.close()

def create_redshift_evolution_of_NL_scalers(
    sim, 
    selection, 
    figpath,
    respath,    
):
    
    catpath_nl_scaled = f'{respath}run_20260311_NL_scaling/IA_fitting_results_summary_{selection}.csv'

    df_nl_scaled = pd.read_csv(catpath_nl_scaled, sep='\t')

    df_nl_scaled_projections = df_nl_scaled[(df_nl_scaled['simulation'] == sim) & (df_nl_scaled['estimator'] == 'projections')]
    df_nl_scaled_multipoles = df_nl_scaled[(df_nl_scaled['simulation'] == sim) & (df_nl_scaled['estimator'] == 'multipoles')]

    fig, axes = plt.subplots(2, 2, figsize=(11, 10))
    axes_flattened = axes.flatten()

    param_list = ['A_IA', 'b_g', 'alpha_NLgg', 'alpha_NLgp']
    label_list = [r'$A_{\rm IA}$', r'$b_{\rm g}$', r'$\alpha^{\rm NL}_{\rm gg}$', r'$\alpha^{\rm NL}_{\rm gp}$']

    for ax_idx in range(4):
        param = param_list[ax_idx]
        label = label_list[ax_idx]
        axes_flattened[ax_idx].errorbar(df_nl_scaled_projections['redshift'], df_nl_scaled_projections[param], yerr=df_nl_scaled_projections[f'{param}_err'], fmt='o', label='Projections')
        axes_flattened[ax_idx].errorbar(df_nl_scaled_multipoles['redshift'], df_nl_scaled_multipoles[param], yerr=df_nl_scaled_multipoles[f'{param}_err'], fmt='s', label='Multipoles')
        axes_flattened[ax_idx].set_xlabel('Redshift')
        axes_flattened[ax_idx].set_ylabel(label)
        axes_flattened[ax_idx].legend(loc='upper left')

    plt.suptitle(f'Comparison of NL scaling - {sim} - {selection}', y=0.91)
    outpath = f'{figpath}/comparison_NL_scaling_{selection}_{sim}.png'
    plt.savefig(outpath, bbox_inches='tight')
    plt.close()

# Save comparison contour plot of MCMCs

In [5]:
sim = 'L400_m7'
probe = 'DM'
estimator = 'multipoles'

figpath = '/home/dneup16/leiden_phd/scripts/results/IA_redshift_dependency_simulations/comparison_plots/mcmc_results/'
os.makedirs(figpath, exist_ok=True)
respath = '/home/dneup16/leiden_phd/scripts/results/IA_redshift_dependency_simulations/'

snapshot_list = [68, 76, 84, 92, 102, 127]
estimator_list = ['multipoles', 'projections']
probe_list = ['DM', 'stars']
redshift_dict = {'68': 2.5, '76': 2.0, '84': 1.5, '92': 1.0, '102': 0.5, '127': 0.0}
for probe in probe_list:
    selection = f'{probe}_nstar_gt50_mstar_gt9p27_mDM_gt11p34'
    for estimator in estimator_list:
        for snapshot in snapshot_list:
            create_contour_comparison_plot(
                snapshot=snapshot,
                sim=sim,
                selection=selection,
                estimator=estimator,
                redshift_dict=redshift_dict,
                figpath=figpath,
                respath=respath,
            )

# Plot comparison redshift evolution of different parameters

In [6]:
sim = 'L400_m7'

figpath = '/home/dneup16/leiden_phd/scripts/results/IA_redshift_dependency_simulations/comparison_plots/redshift_evolution/'
figpath_nl_scaling = '/home/dneup16/leiden_phd/scripts/results/IA_redshift_dependency_simulations/comparison_plots/NL_scaling/'
os.makedirs(figpath, exist_ok=True)
os.makedirs(figpath_nl_scaling, exist_ok=True)
probe_list = ['DM', 'stars']
estimator_list = ['multipoles', 'projections']
for probe in probe_list:
    selection = f'{probe}_nstar_gt50_mstar_gt9p27_mDM_gt11p34'
    for estimator in estimator_list:
        create_redshift_evolution_comparison_plot(
            sim=sim,
            selection=selection,
            estimator=estimator,
            figpath=figpath,
            respath=respath,
        )

    create_redshift_evolution_of_NL_scalers(
        sim=sim,
        selection=selection,
        figpath=figpath_nl_scaling,
        respath=respath,
    )

# Compare IA models over fitting ranges

## Helper functions

In [7]:
def create_fit_window_variation_plot(
    sim,
    selection,
    estimator,
    redshift,
    rmin,
    df_vanilla_list,
    df_NL_scaled_list,
    figfile,
):
    A_IA_vanilla_list = []
    A_IA_err_vanilla_list = []
    b_g_vanilla_list = []
    b_g_err_vanilla_list = []
    red_chi2_vanilla_list = []

    A_IA_NL_scaled_list = []
    A_IA_err_NL_scaled_list = []
    b_g_NL_scaled_list = []
    b_g_err_NL_scaled_list = []
    red_chi2_NL_scaled_list = []

    A_IA_err_ratio_list = []
    b_g_err_ratio_list = []

    for df_idx in range(len(rmin)):
        df_vanilla = df_vanilla_list[df_idx]
        df_NL_scaled = df_NL_scaled_list[df_idx]

        df_vanilla_cut = df_vanilla[(df_vanilla['simulation'] == sim) & (df_vanilla['estimator'] == estimator) & (df_vanilla['redshift'] == redshift)]
        df_NL_scaled_cut = df_NL_scaled[(df_NL_scaled['simulation'] == sim) & (df_NL_scaled['estimator'] == estimator) & (df_NL_scaled['redshift'] == redshift)]

        A_IA_vanilla = df_vanilla_cut['A_IA'].values[0]
        A_IA_err_vanilla = df_vanilla_cut['A_IA_err'].values[0]
        b_g_vanilla = df_vanilla_cut['b_g'].values[0]
        b_g_err_vanilla = df_vanilla_cut['b_g_err'].values[0]
        red_chi2_vanilla = df_vanilla_cut['reduced_chi2'].values[0]

        A_IA_NL_scaled = df_NL_scaled_cut['A_IA'].values[0]
        A_IA_err_NL_scaled = df_NL_scaled_cut['A_IA_err'].values[0]
        b_g_NL_scaled = df_NL_scaled_cut['b_g'].values[0]
        b_g_err_NL_scaled = df_NL_scaled_cut['b_g_err'].values[0]
        red_chi2_NL_scaled = df_NL_scaled_cut['reduced_chi2'].values[0]

        A_IA_vanilla_list.append(A_IA_vanilla)
        A_IA_err_vanilla_list.append(A_IA_err_vanilla)
        b_g_vanilla_list.append(b_g_vanilla)
        b_g_err_vanilla_list.append(b_g_err_vanilla)
        red_chi2_vanilla_list.append(red_chi2_vanilla)

        A_IA_NL_scaled_list.append(A_IA_NL_scaled)
        A_IA_err_NL_scaled_list.append(A_IA_err_NL_scaled)
        b_g_NL_scaled_list.append(b_g_NL_scaled)
        b_g_err_NL_scaled_list.append(b_g_err_NL_scaled)
        red_chi2_NL_scaled_list.append(red_chi2_NL_scaled)

        A_IA_err_ratio = np.nan if A_IA_err_vanilla == 0 else A_IA_err_NL_scaled / A_IA_err_vanilla
        b_g_err_ratio = np.nan if b_g_err_vanilla == 0 else b_g_err_NL_scaled / b_g_err_vanilla
        A_IA_err_ratio_list.append(A_IA_err_ratio)
        b_g_err_ratio_list.append(b_g_err_ratio)

    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    axes = axes.flatten()

    # A_IA
    axes[0].errorbar(
        rmin,
        A_IA_vanilla_list,
        yerr=A_IA_err_vanilla_list,
        fmt='o',
        color='blue',
        linestyle='-',
        label='Vanilla NLA',
        capsize=4,
        capthick=1.2,
    )
    axes[0].errorbar(
        rmin,
        A_IA_NL_scaled_list,
        yerr=A_IA_err_NL_scaled_list,
        fmt='s',
        color='red',
        linestyle='--',
        label='N+LA',
        capsize=4,
        capthick=1.2,
    )
    axes[0].set_xlabel('rmin [Mpc/h]')
    axes[0].set_ylabel(r'$A_{\rm IA}$')
    axes[0].set_xscale('log')
    axes[0].set_xlim(rmin[0] * 0.8, rmin[-1] * 1.2)
    axes[0].legend(loc='upper right')

    # b_g
    axes[1].errorbar(
        rmin,
        b_g_vanilla_list,
        yerr=b_g_err_vanilla_list,
        fmt='o',
        color='blue',
        linestyle='-',
        label='Vanilla NLA',
        capsize=4,
        capthick=1.2,
    )
    axes[1].errorbar(
        rmin,
        b_g_NL_scaled_list,
        yerr=b_g_err_NL_scaled_list,
        fmt='s',
        color='red',
        linestyle='--',
        label='N+LA',
        capsize=4,
        capthick=1.2,
    )
    axes[1].set_xlabel('rmin [Mpc/h]')
    axes[1].set_ylabel(r'$b_{\rm g}$')
    axes[1].set_xscale('log')
    axes[1].set_xlim(rmin[0] * 0.8, rmin[-1] * 1.2)
    axes[1].legend(loc='upper right')
    
    # Reduced chi-squared
    axes[2].plot(rmin, red_chi2_vanilla_list, marker='o', color='blue', linestyle='-', label='Vanilla NLA')
    axes[2].plot(rmin, red_chi2_NL_scaled_list, marker='s', color='red', linestyle='--', label='N+LA')
    axes[2].set_xlabel('rmin [Mpc/h]')
    axes[2].set_ylabel(r'$\chi^2_{\rm red}$')
    axes[2].set_xscale('log')
    axes[2].hlines(1, rmin[0], rmin[-1], color='gray', linestyle='--', label=r'$\chi^2_{\rm red}=1$')
    axes[2].set_ylim(0, 5)
    axes[2].set_xscale('log')
    axes[2].set_xlim(rmin[0] * 0.8, rmin[-1] * 1.2)
    axes[2].legend(loc='upper right')

    # Ratio of A_IA errorbars: NL scaled / vanilla
    axes[3].plot(rmin, A_IA_err_ratio_list, marker='d', color='black', linestyle='-')
    axes[3].set_xlabel('rmin [Mpc/h]')
    axes[3].set_ylabel(r'$\sigma(A_{\rm IA})_{\rm N+LA} / \sigma(A_{\rm IA})_{\rm Vanilla}$')
    axes[3].set_xscale('log')
    axes[3].set_xlim(rmin[0] * 0.8, rmin[-1] * 1.2)
    axes[3].hlines(1, rmin[0], rmin[-1], color='gray', linestyle='--')

    # Ratio of b_g errorbars: NL scaled / vanilla
    axes[4].plot(rmin, b_g_err_ratio_list, marker='d', color='black', linestyle='-')
    axes[4].set_xlabel('rmin [Mpc/h]')
    axes[4].set_ylabel(r'$\sigma(b_{\rm g})_{\rm N+LA} / \sigma(b_{\rm g})_{\rm Vanilla}$')
    axes[4].set_xscale('log')
    axes[4].set_xlim(rmin[0] * 0.8, rmin[-1] * 1.2)
    axes[4].hlines(1, rmin[0], rmin[-1], color='gray', linestyle='--')

    # Add fiducial fitting range vline to axes 0,1,3,4
    fiducial_rmin = 6.0  # Mpc/h
    for ax_ in axes:
        ax_.axvline(fiducial_rmin, color='black', linestyle=':')

    plt.suptitle(f'Variation of fit window - (z={redshift}) - {estimator} - {sim} - {selection}', y=1.02)
    plt.savefig(figfile, bbox_inches='tight')
    plt.close()

## Load in data

In [8]:
rmin = np.geomspace(0.1, 20,10)
rmin = rmin[:-1]
respath = '/home/dneup16/leiden_phd/scripts/results/IA_redshift_dependency_simulations/'
figpath = '/home/dneup16/leiden_phd/scripts/results/IA_redshift_dependency_simulations/comparison_plots/fit_window_variation/'
os.makedirs(figpath, exist_ok=True)
sim = 'L400_m7'
probe = 'DM'
estimator = 'projections'
# redshift = 0.5
path_to_vanilla = f'{respath}run_20260311_fitWindowVariation'
path_to_NL_scaled = f'{respath}run_20260311_NL_scaling_fitWindowVariation'
selection = f'{probe}_nstar_gt50_mstar_gt9p27_mDM_gt11p34'

df_vanilla_list = []
df_NL_scaled_list = []
for rmin_val in rmin:
    fitWindow_str = f'fitWindow_{rmin_val:.2f}_50'
    df_vanilla = pd.read_csv(f'{path_to_vanilla}/{fitWindow_str}/IA_fitting_results_summary_{selection}.csv', sep='\t')
    df_NL_scaled = pd.read_csv(f'{path_to_NL_scaled}/{fitWindow_str}/IA_fitting_results_summary_{selection}.csv', sep='\t')
    df_vanilla_list.append(df_vanilla)
    df_NL_scaled_list.append(df_NL_scaled)

    if rmin_val == rmin[0]:
        print(df_vanilla_list[0].columns)

Index(['simulation', 'snapshot', 'redshift', 'A_IA', 'A_IA_err', 'b_g',
       'b_g_err', 'reduced_chi2', 'estimator'],
      dtype='object')


## Plot A_IA and b_g over fitting range

In [9]:
sim = 'L400_m7'
redshift_list = [2.5, 2.0, 1.5, 1.0, 0.5, 0.0]
estimator_list = ['multipoles', 'projections']

for estimator in estimator_list:
    for redshift in redshift_list:
        figfile = f'{figpath}/fit_window_variation_{selection}_{estimator}_{sim}_redshift_{redshift}.png'
        create_fit_window_variation_plot(
            sim=sim,
            selection=selection,
            estimator=estimator,
            redshift=redshift,
            rmin=rmin,
            df_vanilla_list=df_vanilla_list,
            df_NL_scaled_list=df_NL_scaled_list,
            figfile=figfile,
        )